In [1]:
from pathlib import Path
import pandas as pd

from m5_forecasting.features.encoding import encode_categorical_features
from m5_forecasting.features.scaling import scale_features
from m5_forecasting.data.sequence import create_sequences
from m5_forecasting.data.split import train_val_test_split
from m5_forecasting.models.lstm import (
    build_lstm_model,
    get_callbacks
)

I0000 00:00:1783307251.947064   61262 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1783307252.041602   61262 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783307255.084465   61262 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/processed")

final_data = pd.read_parquet(
    DATA_PATH / "final_data.parquet"
)

In [3]:
final_data = final_data.dropna().reset_index(drop=True)

In [4]:
final_data, encoders = encode_categorical_features(final_data)

In [5]:
final_data, scaler = scale_features(final_data)

In [6]:
feature_columns = [
    "sell_price",
    "lag_1",
    "lag_7",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "month",
    "day",
    "day_of_week",
    "week_of_year",
    "is_weekend",
]

target_column = "sales"

In [7]:
X, y = create_sequences(
    final_data,
    feature_columns,
    target_column,
    sequence_length=28
)

In [8]:
(
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,
    y_test,
) = train_val_test_split(X, y)

In [9]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(68, 28, 11)
(15, 28, 11)
(15, 28, 11)


In [10]:
print("Training sequences:", X_train.shape)
print("Validation sequences:", X_val.shape)
print("Test sequences:", X_test.shape)

if X_train.ndim != 3 or X_train.shape[1] == 0:
    raise ValueError("Expected 3D sequence data with at least one time step per sample.")

model = build_lstm_model(
    input_shape=(
        X_train.shape[1],
        X_train.shape[2]
    )
)

model.summary()

Training sequences: (68, 28, 11)
Validation sequences: (15, 28, 11)
Test sequences: (15, 28, 11)


E0000 00:00:1783307258.037569   61262 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/ibrahimkhan/m5-sales-forecasting/.venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        19,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,569 (84.25 KB)

 Trainable params: 21,569 (84.25 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
callbacks = get_callbacks()

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=12,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 418ms/step - loss: 8.1409 - mae: 1.4429 - val_loss: 1.5823 - val_mae: 0.9140
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 87ms/step - loss: 7.9495 - mae: 1.4420 - val_loss: 1.4504 - val_mae: 0.8960
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - loss: 7.6846 - mae: 1.4456 - val_loss: 1.3186 - val_mae: 0.8764
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - loss: 7.3696 - mae: 1.4371 - val_loss: 1.1843 - val_mae: 0.8547
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - loss: 7.1000 - mae: 1.4531 - val_loss: 1.0416 - val_mae: 0.8256
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - loss: 6.6260 - mae: 1.4453 - val_loss: 0.9045 - val_mae: 0.7834
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - loss: 6.2082 - mae: 1.4528 - val_loss: 0.8661 - val_mae: 0.7541
Epoch 8/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - loss: 5.9840 - mae: 1.5371 - val_loss: 1.2769 - val_mae: 1.0021
Epoch 9/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - loss: 5.8949 - mae: 1.7143 -

In [13]:
test_loss, test_mae = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test MAE : {test_mae:.4f}")

Test Loss: 1.1843
Test MAE : 0.9879


In [14]:
predictions = model.predict(X_test)

comparison = pd.DataFrame({
    "Actual": y_test,
    "Predicted": predictions.flatten()
})

comparison.head(20)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 506ms/step


,Actual,Predicted
0,0,0.999293
1,1,1.153047
2,3,0.959071
3,0,0.817325
4,0,1.001237
5,0,1.014298
6,2,1.061761
7,0,0.953161
8,0,1.026443
9,0,1.092597
